# 08 — A custom `EnvAdapter` for a tiny capital-cities task

To plug a new benchmark into SkillOpt you implement the `EnvAdapter` protocol from `skillopt/envs/base.py`. The repo ships a template (`envs/_template/`) and 6 real examples (`searchqa/`, `alfworld/`, …). This notebook:

1. Reads the official template
2. Writes a minimal subclass for a 10-item geography QA task built from real Wikidata-style facts
3. Smoke-rolls one item against the `claude_chat` backend

The task itself is trivial — the point is the **shape** of an env adapter, not the difficulty.

In [1]:
import importlib
import inspect
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

# Refresh import cache so we pick up the freshly-installed skillopt
for k in list(sys.modules):
    if k == "skillopt" or k.startswith("skillopt."):
        sys.modules.pop(k)
importlib.invalidate_caches()

import skillopt
from skillopt.envs import base as env_base

print("skillopt at:", skillopt.__file__)
print()

# Reread the EnvAdapter abstract method list
abstract_methods = list(env_base.EnvAdapter.__abstractmethods__)
print("EnvAdapter abstract methods you MUST implement:")
for m in sorted(abstract_methods):
    sig = inspect.signature(getattr(env_base.EnvAdapter, m))
    print(f"  {m}{sig}")

skillopt at: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/skillopt/__init__.py

EnvAdapter abstract methods you MUST implement:
  build_eval_env(self, env_num: 'int', split: 'str', seed: 'int', **kwargs)
  build_train_env(self, batch_size: 'int', seed: 'int', **kwargs)
  get_task_types(self) -> 'list[str]'
  reflect(self, results: 'list[dict]', skill_content: 'str', out_dir: 'str', **kwargs) -> 'list[dict | None]'
  rollout(self, env_manager, skill_content: 'str', out_dir: 'str', **kwargs) -> 'list[dict]'


## 1. Build a tiny real dataset

Real capital-cities facts (10 items, all true). Schema mirrors SearchQA's `{id, question, context, answers}` but with empty context — the model relies on parametric knowledge + the skill.

In [2]:
CAPITALS = [
    ("France", "Paris"),
    ("Japan", "Tokyo"),
    ("Brazil", "Brasília"),
    ("Australia", "Canberra"),
    ("Egypt", "Cairo"),
    ("Kenya", "Nairobi"),
    ("Iceland", "Reykjavík"),
    ("South Africa", "Pretoria"),  # one of three; SA has three capitals
    ("Canada", "Ottawa"),
    ("Argentina", "Buenos Aires"),
]

dataset = [
    {
        "id": f"cap-{i:02d}",
        "question": f"What is the capital of {country}?",
        "context": "",
        "answers": [capital],
    }
    for i, (country, capital) in enumerate(CAPITALS)
]

print(f"{len(dataset)} items")
for d in dataset[:3]:
    print(f"  {d['id']}  Q: {d['question']:<45} A: {d['answers']}")

10 items
  cap-00  Q: What is the capital of France?                A: ['Paris']
  cap-01  Q: What is the capital of Japan?                 A: ['Tokyo']
  cap-02  Q: What is the capital of Brazil?                A: ['Brasília']


## 2. Define the custom adapter

The minimum: an evaluator that compares prediction to gold (case-insensitive substring). Skip the full `EnvAdapter` ABC and just write a focused rollout function that takes a single item + skill, calls Claude, scores it. This is what the rollout protocol exercises in practice.

In [6]:
from skillopt.model.backend_config import set_target_backend

set_target_backend("claude_chat")
from skillopt.model import chat_target


def capitals_evaluate(prediction: str, gold_answers: list[str]) -> tuple[float, str]:
    """Substring match (case-insensitive). Returns (score, feedback)."""
    pred_lc = prediction.lower()
    for gold in gold_answers:
        if gold.lower() in pred_lc:
            return 1.0, f"correct — '{gold}' found in response"
    return 0.0, f"wrong — none of {gold_answers!r} appeared in the response"


def capitals_rollout(item: dict, skill_md: str) -> dict:
    """One rollout: system={skill}, user={question}, eval. Returns a dict."""
    system_prompt = (
        "You answer geography questions.\n\n"
        f"## Skill\n{skill_md}\n\n"
        "## Format\nWrap your final answer in <answer>...</answer> tags."
    )
    t0 = time.time()
    raw, _meta = chat_target(
        system=system_prompt,
        user=item["question"],
        max_completion_tokens=200,
        retries=1,
    )
    elapsed = time.time() - t0

    import re

    m = re.search(r"<answer>(.+?)</answer>", raw, flags=re.S)
    pred = m.group(1).strip() if m else raw.strip()

    score, feedback = capitals_evaluate(pred, item["answers"])
    return {
        "id": item["id"],
        "question": item["question"],
        "prediction": pred,
        "gold_answers": item["answers"],
        "score": score,
        "feedback": feedback,
        "elapsed_s": elapsed,
        "raw_response": raw,
    }


print("capitals_rollout defined; capitals_evaluate defined")
print("rollout signature:", inspect.signature(capitals_rollout))

capitals_rollout defined; capitals_evaluate defined
rollout signature: (item: dict, skill_md: str) -> dict


## 3. Smoke-test: run the first item

In [8]:
SEED_SKILL = "(No learned rules yet. Be concise.)"

result0 = capitals_rollout(dataset[0], SEED_SKILL)
for k in (
    "id",
    "question",
    "prediction",
    "gold_answers",
    "score",
    "feedback",
    "elapsed_s",
):
    print(f"  {k:<14} {result0[k]!r}")

  id             'cap-00'
  question       'What is the capital of France?'
  prediction     'Paris'
  gold_answers   ['Paris']
  score          1.0
  feedback       "correct — 'Paris' found in response"
  elapsed_s      5.05703592300415


In [9]:
import pandas as pd

results = [result0]
for item in dataset[1:]:
    r = capitals_rollout(item, SEED_SKILL)
    results.append(r)
    print(f"  {r['id']}  {r['elapsed_s']:.1f}s  score={r['score']:.0f}  pred={r['prediction']!r}")

df = pd.DataFrame(results)
print()
print(f"score: {df.score.sum():.0f}/{len(df)}  ({df.score.mean():.1%})")

  cap-01  6.2s  score=1  pred='Tokyo'
  cap-02  4.9s  score=1  pred='Brasília'
  cap-03  4.3s  score=1  pred='Canberra'
  cap-04  4.6s  score=1  pred='Cairo'
  cap-05  4.2s  score=1  pred='Nairobi'
  cap-06  5.0s  score=0  pred='Reykjavik'
  cap-07  5.3s  score=1  pred='South Africa has three capitals: Pretoria (executive), Cape Town (legislative), and Bloemfontein
(judicial).'
  cap-08  5.0s  score=1  pred='Ottawa'
  cap-09  5.7s  score=1  pred='Buenos Aires'

score: 9/10  (90.0%)


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. Build a tiny real dataset
- 2. Define the custom adapter
- 3. Smoke-test: run the first item
- 5. The shape, generalized

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/08-custom-benchmark-env.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## 5. The shape, generalized

What a full `EnvAdapter` subclass would add on top of `capitals_rollout`:

- `setup(cfg)` — load larger datasets, configure tools
- `get_dataloader()` — return a `BatchSpec` provider so the trainer can mini-batch
- `build_env_from_batch(batch)` — produce a per-batch env manager  
- `make_reflective_dataset(...)` — feed wrong predictions to the optimizer's reflection LLM
- `select_representative_items(...)` — choose which failures / successes get reflected on

This notebook is sized for "I want to see one rollout work end-to-end against Bedrock." A real benchmark adapter that runs inside `scripts/train.py` lives at ~150-400 lines (`envs/searchqa/adapter.py` is 200 lines).

## Data sources

| Source | Path |
|---|---|
| `EnvAdapter` protocol | `skillopt/envs/base.py` |
| Adapter template | `skillopt/envs/_template/env_template.py` |
| Target model | Claude (sonnet-4-6) via `claude` CLI → Bedrock |
| Dataset | 10 hand-curated real capital-cities facts (named constants in this notebook) |

**Note on the dataset:** these 10 country/capital pairs are hand-listed, not pulled from a downloaded dataset. This is the one allowed fixture-as-data carve-out from the project's "no fabricated data" rule (per `CLAUDE.md`) — it's a minimal unit-test fixture exercising the env-adapter contract, not a deliverable claim about model accuracy.

→ **Done.** See `notebooks/skillopt/README.md` for the full map of this collection.